# 02 · `evosax`: one loop for **every** algorithm

**What this notebook is.** A single, algorithm-agnostic driver that runs any
[`evosax`](https://github.com/RobertTLange/evosax) strategy against the origami
folding simulation. `evosax` provides the ES/GA/PSO implementations; this
library provides the *search space* and the *objective*. Even this notebook
contains no search logic of its own — only the `ask → evaluate → tell` glue.

**What you will do**

1. build a small angle-only design space and a `genome -> loss` function;
2. instantiate strategies through the `evosax.algorithms.algorithms` registry;
3. dispatch the three `init` signatures (`mean`, `population + fitness`,
   `means` for the Stein-variational strategies) automatically;
4. run a few named strategies, then **all** of them, and compare;
5. checkpoint the winner and read it back as a crease pattern.

> ### Direction gotcha (read this once)
>
> `evosax` **minimizes**: `state.best_fitness` is the *lowest* value seen.
> `to_evosax_fitness(evaluator)` returns the library's **higher-is-better**
> fitness, so negate it before `tell`:
> `es.tell(key, X, -to_evosax_fitness(evaluator)(genomes), state, params)`.
> In this notebook the sign lives in exactly one place, `batch_loss()`.

> ### Objective mode
>
> The same CPU-cost problem as notebook 01: one MJX-JAX simulation of this
> 6-face model is ~12 s. The loops therefore default to a cheap analytic
> **surrogate** with the identical interface and re-score the winner with the
> real `OrigamiEvaluator`. Set `OBJECTIVE_MODE = "physics"` on a GPU runtime to
> optimize forward displacement for real.


## Install

`evosax` is not a dependency of the library; this notebook installs it
alongside the JAX extra. After the integration branch is merged, replace
`@feat/optimization-integrations-v1` with `@main`.


In [ ]:
%pip install -q "origami-engine-mjx[jax] @ git+https://github.com/ihsan31415/origami-engine-mjx.git@feat/optimization-integrations-v1"
%pip install -q "evosax>=0.3"

In [ ]:
import importlib.metadata
import inspect

import jax
import jax.numpy as jnp
import numpy as np
from evosax.algorithms import algorithms as EVO_ALGORITHMS

from origami_engine_mjx import Crease
from origami_engine_mjx.integrations import (
    DesignSpace,
    EvalRecord,
    OrigamiEvaluator,
    RunLogger,
    save_best,
    save_genome,
)
from origami_engine_mjx.integrations.adapters import space_spec

print("jax          :", jax.__version__, "| devices:", jax.devices())
print("evosax       :", importlib.metadata.version("evosax"))
print("strategies   :", len(EVO_ALGORITHMS))
print("names        :", ", ".join(sorted(EVO_ALGORITHMS))[:400], "...")

## 0. Configure

`RUN_ALL = True` is cheap in surrogate mode (each generation is a few matrix
operations). In physics mode it would cost `pop × generations` simulations per
strategy, so it is disabled there.


In [ ]:
OBJECTIVE_MODE = "surrogate"  # "surrogate" (CPU-fast) or "physics" (real scores)
USE_DEFAULT_PATTERN = False  # True -> packaged 4x4 Miura, 15 variables
POP_SIZE = 4
GENERATIONS = 2
NUM_POPULATIONS = 2  # only used by SV_CMA_ES / SV_Open_ES
SEED = 0
RUN_ALL = OBJECTIVE_MODE == "surrogate"  # all 37 strategies, cheaply
DEMO = ["CMA_ES", "Open_ES", "SimpleGA", "PSO", "DifferentialEvolution"]

EVAL_DURATION = 0.004
EVAL_FOLD = 0.003
print(f"mode={OBJECTIVE_MODE}  pop={POP_SIZE}  generations={GENERATIONS}  run_all={RUN_ALL}")

## 1. Space and real objective

The 2×2 Miura pattern has 6 creases → 4 actuated hinges. `evosax` works in
`R^dim`; we project every candidate into the space's normalized `[-1, 1]` view
with `space.from_normalized` (which clips to bounds). The search space's
`lower` / `upper` therefore define the feasible box.


In [ ]:
if USE_DEFAULT_PATTERN:
    space = DesignSpace.from_default_pattern()
else:
    pattern = [
        Crease((0.1, 0.1), (0.1, 0.5), 0.8),
        Crease((0.1, 0.5), (0.5, 0.5), -0.8),
        Crease((0.5, 0.5), (0.5, 0.1), 0.8),
        Crease((0.5, 0.1), (0.1, 0.1), -0.8),
        Crease((0.1, 0.1), (0.5, 0.5), -0.8),
        Crease((0.1, 0.5), (0.5, 0.1), 0.8),
    ]
    space = DesignSpace.angles_only(pattern, name="miura-2x2")

spec = space_spec(space)
print(f"space {space.name!r}: dim={spec['dim']}  n_creases={spec['n_creases']}")
print("bounds lower:", np.round(spec["lower"], 3).tolist())
print("bounds upper:", np.round(spec["upper"], 3).tolist())

evaluator = OrigamiEvaluator(
    space,
    backend="jax",
    duration=EVAL_DURATION,
    fold_duration=EVAL_FOLD,
    vibrate=False,
    maximize=True,  # fitness = +x centroid displacement
    error_policy="penalty",
)
print(evaluator)

## 2. The `genome -> loss` glue

Two conversions and one sign:

* `z_to_genomes(Z)` — normalized vectors → in-bounds genomes;
* `batch_loss(Z)` — the value `evosax` minimizes.

In physics mode `evaluate_z` returns `-evaluator.evaluate_batch(...).fitness`
(equivalently `-to_evosax_fitness(evaluator)(genomes)`), plus the `BatchResult`
so the run logger never re-simulates. In surrogate mode we minimize the squared
distance to a target design, so the run finishes on CPU.


In [ ]:
target_z = np.array([0.6, -0.6, 0.6, -0.6])[: space.dim]


def clip_z(z):
    return np.clip(np.asarray(z, dtype=float), -1.0, 1.0)


def z_to_genomes(Z):
    """Project evosax vectors into the feasible genome box."""
    return np.vstack([space.from_normalized(row) for row in clip_z(Z)])


def evaluate_z(Z):
    """Return (genomes, per-individual loss, BatchResult or None).

    The real evaluation happens exactly once per population. The
    ``BatchResult`` is only kept in physics mode so the run logger can
    reuse it instead of paying for a second simulation.
    """
    Z = clip_z(Z)
    genomes = z_to_genomes(Z)
    if OBJECTIVE_MODE == "physics":
        result = evaluator.evaluate_batch(genomes, return_details=False)
        return genomes, -np.asarray(result.fitness, dtype=float), result
    return genomes, np.mean((Z - target_z) ** 2, axis=1), None


def batch_loss(Z):
    """Return the per-individual loss that evosax MINIMIZES."""
    return evaluate_z(Z)[1]


Z_demo = space.sample_normalized(np.random.default_rng(0), size=3)
print("Z_demo      :", np.round(Z_demo, 3).tolist())
print("genomes     :", np.round(z_to_genomes(Z_demo), 3).tolist())
print("losses      :", np.round(batch_loss(Z_demo), 6).tolist())
print("target loss :", round(float(batch_loss(target_z[None, :])[0]), 9))

## 3. Construct any strategy

Every class in `evosax.algorithms.algorithms` shares the
`init(key, ...) / ask(key, state, params) / tell(key, pop, fitness, state, params)`
interface, but constructors and `init` signatures differ in three ways:

| Signature | Strategies | `init` arguments |
|---|---|---|
| distribution-based | CMA-ES, Open-ES, xNES, PGPE, … | `init(key, mean, params)` |
| population-based | SimpleGA, PSO, DifferentialEvolution, … | `init(key, population, fitness, params)` |
| Stein-variational | `SV_CMA_ES`, `SV_Open_ES` | `init(key, means, params)`, needs `num_populations=` |
| explicit sampler | `RandomSearch` | needs `sampling_fn=` |

The helpers below detect the signature with `inspect` instead of hard-coding a
list, so **new evosax strategies keep working**.


In [ ]:
def make_strategy(name, dim, population_size, num_populations=NUM_POPULATIONS):
    """Construct an evosax strategy with the extra kwargs it may require."""
    cls = EVO_ALGORITHMS[name]
    template = jnp.zeros((dim,), dtype=jnp.float32)
    kwargs = {}
    if name == "RandomSearch":
        kwargs["sampling_fn"] = lambda key: jax.random.uniform(key, (dim,), minval=-1.0, maxval=1.0, dtype=jnp.float32)
    if name in {"SV_CMA_ES", "SV_Open_ES"}:
        kwargs["num_populations"] = num_populations
    return cls(population_size=population_size, solution=template, **kwargs)


def init_state(es, key, seed):
    """Dispatch on the strategy's init signature."""
    params = es.default_params
    k_init, key = jax.random.split(key)
    init_args = inspect.signature(es.init).parameters

    if "population" in init_args:  # population-based
        Z0 = space.sample_normalized(np.random.default_rng(seed), size=es.population_size)
        F0 = batch_loss(Z0)
        state = es.init(k_init, jnp.asarray(Z0), jnp.asarray(F0, dtype=jnp.float32), params)
    elif "means" in init_args:  # Stein-variational
        means = jnp.zeros((es.num_populations, space.dim), dtype=jnp.float32)
        state = es.init(k_init, means, params)
    else:  # distribution-based
        state = es.init(k_init, jnp.zeros((space.dim,), dtype=jnp.float32), params)
    return state, params, key


es = make_strategy("CMA_ES", space.dim, POP_SIZE)
print(type(es).__module__ + "." + type(es).__name__)
print("population_size:", es.population_size, "| num_dims:", es.num_dims)
print("default_params :", type(es.default_params).__name__)
print("init args      :", list(inspect.signature(es.init).parameters))

## 4. The universal loop

Same body for every strategy:

```
ask -> clip to [-1, 1] -> evaluate (loss) -> tell
```

We keep our own best-so-far because some strategies' archives are
per-population (`SV_*`); the archive is still reported. `tell` receives the
**clipped** population that was actually evaluated, not the raw samples.


In [ ]:
def run_evosax(name, generations=GENERATIONS, population_size=POP_SIZE, seed=SEED, log=None, verbose=False):
    es = make_strategy(name, space.dim, population_size)
    state, params, key = init_state(es, jax.random.key(seed), seed)

    history, best_z, best_loss = [], None, np.inf
    for generation in range(generations):
        key, k_ask, k_tell = jax.random.split(key, 3)

        Z, state = es.ask(k_ask, state, params)
        genomes, losses, result = evaluate_z(np.asarray(Z))  # one evaluation
        Z = jnp.asarray(clip_z(np.asarray(Z)), dtype=jnp.float32)  # what we evaluated
        losses = np.asarray(losses, dtype=float)

        state, _metrics = es.tell(k_tell, Z, jnp.asarray(losses, dtype=jnp.float32), state, params)

        i = int(np.argmin(losses))
        if losses[i] < best_loss:
            best_z, best_loss = np.asarray(Z[i]), float(losses[i])
        history.append(float(np.min(losses)))

        if log is not None:
            if result is not None:  # physics: reuse, never re-simulate
                log.log_batch(result, genomes)
            else:
                for genome, loss in zip(genomes, losses, strict=True):
                    log.log(EvalRecord(fitness=-loss, raw=-loss, path="surrogate"), genome)

        if verbose:
            print(f"  gen {generation + 1:2d}  best loss={history[-1]:+.6f}  mean={np.mean(losses):+.6f}")

    best_genome = space.from_normalized(best_z)
    return {
        "algorithm": name,
        "best_loss": best_loss,
        "best_genome": best_genome,
        "best_z": best_z,
        "history": history,
        "archive_fitness": np.asarray(state.best_fitness).reshape(-1),
        "state": state,
    }


demo = {}
with RunLogger("runs/evosax_demo.jsonl") as log:
    for name in DEMO:
        print(name)
        demo[name] = run_evosax(name, log=log, verbose=True)

print("\nstrategy                      best loss   generations")
for name, out in demo.items():
    print(f"{name:28s} {out['best_loss']:+.6f}   {np.round(out['history'], 4).tolist()}")

## 5. Run **every** registered strategy

Because the surrogate is analytic, all strategies run in seconds. Failures are
caught per strategy so one unsupported constructor cannot abort the notebook —
if you find a new one, add its required kwargs in `make_strategy`.

Switch `OBJECTIVE_MODE = "physics"` (GPU) and `RUN_ALL = False` to run this
against the simulation; then the budget is `pop × generations` full simulations
per strategy, plus `pop` for the population-based `init`.


In [ ]:
all_results, failures = {}, {}
if RUN_ALL:
    for name in sorted(EVO_ALGORITHMS):
        try:
            all_results[name] = run_evosax(name, generations=GENERATIONS)
        except Exception as exc:  # noqa: BLE001 - survey mode
            failures[name] = f"{type(exc).__name__}: {exc}"

    rows = sorted(all_results.items(), key=lambda kv: kv[1]["best_loss"])
    print(f"ran {len(all_results)}/{len(EVO_ALGORITHMS)} strategies\n")
    print(f"{'strategy':24s} {'best loss':>12s}  {'archive (min)':>13s}")
    for name, out in rows:
        archive = float(np.nanmin(out["archive_fitness"])) if out["archive_fitness"].size else float("nan")
        print(f"{name:24s} {out['best_loss']:12.6f}  {archive:13.6f}")

    if failures:
        print("\nfailures:")
        for name, err in failures.items():
            print(f"  {name:24s} {err[:100]}")
else:
    print("RUN_ALL is False (physics mode). Set RUN_ALL = True in surrogate mode.")

## 6. Score the winner with the real physics and checkpoint it

This cell is the hand-off point. In physics mode the winner's score *is* the
optimization objective; in surrogate mode it is a wiring check — the discovered
design is real, the ranking was done on the stand-in.


In [ ]:
candidates = list(all_results.values()) if all_results else list(demo.values())
winner = min(candidates, key=lambda out: out["best_loss"])
best_genome = winner["best_genome"]

record = evaluator.evaluate(best_genome)  # one real simulation
path = save_best(
    best_genome,
    space,
    "runs/evosax_best.json",
    fitness=record.fitness,
    extra={"algorithm": winner["algorithm"], "objective_mode": OBJECTIVE_MODE},
)
crease_path = save_genome(best_genome, space, "runs/evosax_best_pattern.json")

print(f"winner        : {winner['algorithm']}  (loss {winner['best_loss']:+.6f})")
print(f"physics score : fitness={record.fitness:+.6f} m  status={record.status}")
print(f"checkpoint    : {path}")
print(f"crease file   : {crease_path}")
print("best genome   :", np.round(space.repair(best_genome), 4).tolist())

## 7. Tuning and cost notes

* **Hyperparameters** are a params dataclass: `params = es.default_params.replace(std_init=0.5)`
  (flax struct) and then pass `params` to `init` / `ask` / `tell`.
* **Reproducibility**: `jax.random.key(SEED)`; split it for `init`, `ask` and
  `tell`. Physics is deterministic, so a fixed seed gives a fixed trajectory.
* **Cost** = `population_size × generations` simulations (+`population_size`
  for the population-based `init`). `evosax` itself is JIT-compiled and cheap;
  the simulation dominates by orders of magnitude.
* **Bounds**: evosax samples in unbounded `R^dim`. Clipping to `[-1, 1]` before
  evaluation *and* before `tell` keeps the algorithm's state consistent with
  what was scored. For unbounded strategies, consider a smooth penalty instead
  of clipping.
* **Nothing here is library code.** `origami_engine_mjx` never imports `evosax`;
  this notebook is the integration.

Next: **03** (RL environments) and **04** (AI / LLM agent in the loop).
Docs: `docs/OPTIMIZATION_ES.md` · `docs/INTEGRATIONS.md`.
Reusable recipe: `prompts/origami-optimizer-integration.md`.
